# Lab 9: Build a test set and score the assistant
**Module 5.1: Measuring quality.**  **Time:** 30 minutes.

**Goal:** "it looks good in the demo" is not a quality bar. You will write test questions with expected answers,
then score the RAG assistant two ways:
* **retrieval hit:** did search find the right document?
* **answer verdict:** an LLM judge compares the answer with the expected answer (LLM-as-judge).

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
qc, _ = lk.build_policy_index()

## 1. The starter test set (5 questions)

In [ ]:
import pandas as pd
tests = lk.load_csv("data/eval_starter.csv")
pd.DataFrame(tests)

## 2. Your turn: add 10 questions
Open any PDF in `data/policies` (in Colab: the folder icon on the left, then `course/data/policies`).
Good test sets include: easy facts, numbers, questions needing two passages, questions phrased the way
employees really talk, and **at least two questions the documents cannot answer** (expected answer
`NOT IN DOCUMENTS`, source_doc `none`).

In [ ]:
my_tests = [
    {"question": "What is the return window for electronics?", "expected_answer": "10 days",
     "source_doc": "05_refund_returns_policy.pdf"},
    {"question": "What is Meridian's share price?", "expected_answer": "NOT IN DOCUMENTS", "source_doc": "none"},
    # add 8 more here
]
all_tests = tests + my_tests
print(len(all_tests), "questions")

## 3. How the judge works
Read the judge's instructions. The judge is itself an LLM, so it can be wrong. Spot-check its verdicts.

In [ ]:
print(lk.JUDGE_SYSTEM)
lk.judge("What is the minimum password length?", "14 characters", "Passwords need at least 14 characters [1].")

## 4. Score the assistant

In [ ]:
baseline = lk.evaluate(qc, all_tests, label="baseline")
baseline

## 5. Read the failures
For each FAIL decide: was it **retrieval** (wrong document found), **generation** (right document, wrong answer),
**the judge** (answer was fine), or **the test** (expected answer was unclear)?

In [ ]:
baseline[baseline.verdict == "FAIL"][["question", "retrieval_hit", "answer", "judge_reason"]]

In [ ]:
baseline.to_csv("baseline_scores.csv", index=False)   # Lab 10 compares against this file
print("saved baseline_scores.csv")

**Discuss:** who in your organisation should own the test set: the AI team, the business, or both?
How often should it grow, and from where (real user questions, incidents, complaints)?